# Model_ENSOClim_CoeffCheck-avg

Caroline Juang, c.juang@columbia.edu

July 2024

**This check deals with the average of the current-season gSST and prior-season gSST. This check feeds into the model `Model_ENSOclim_AkaikeCoeff`.**

In [1]:
# import
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.stats import linregress

# User input

In [2]:
# customize seasons for climate variables

# customize number of rolling periods
ant_years = 1 # antecedent years to include (2 antecedent years + current year)
ant_season = 3 # n+1 of months to include in each period (e.g. input 2 would mean 3 months)

firstyear = 1984 # first year of data
finalyear = 2022 # final year of data (should be same as burned area)
time_length = int(finalyear-firstyear+1) # get length of timeseries

# SST gradient: (125-155 deg lon) - nino3.4 (190-240 deg)
# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindnameObs = f.read().strip()
print(climindnameObs +' will be used for the SST gradient')
climindnameDT = 'DeTrend_'+climindnameObs

# importing data strings
directory = 'your_directory'
#data_string = 'data\\'
data_string = 'data//'
#model_string = 'model\\'+climindnameObs+'\\'
model_string = 'model//'+climindnameObs+'//'

patch125-155_nino3-34 will be used for the SST gradient


# Scripts

In [3]:
# average climate variables, within a selected season

def annualDetrend(data):
    """
    This intakes an array of current ecoregion's climate variable of annual averages (data), 
    Output: an array of the yearly data, detrended so the slope of the data is zero.
    Requirements: 
    """

    # detrend the data
    xnum = np.arange(0,len(data))
    reg = linregress(xnum, data)
    m = reg.slope
    b = reg.intercept
    regpredict = m*xnum + b
    # normalize by subtracting the linear regression
    tmpdatanorm = (data - regpredict) + b

    return tmpdatanorm

# Import data
* Ecoregions as manual input
* **climate** from `Data_CreateModelData`
* **SST gradient** from `Data_CreateENSOIndex` &rarr; `Data_CreateModelData`

In [4]:
# create strings of the types of forests, remove #5
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']
province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

In [5]:
# import model data for climate

dfframesall = {}
dfframesfor = {}
dfframesnon = {}

for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climatefull_ecoprovinces_'
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

In [6]:
# read in the SST gradient data

# import
climindseasons83Obs = pd.read_csv(data_string + 'sstgrad_seasonfull_' +climindnameObs+'_82_y.txt').set_index('Unnamed: 0')
climindseasons83DT = pd.read_csv(data_string + 'sstgrad_seasonfull_' +climindnameDT+'_82_y.txt').set_index('Unnamed: 0')

# create prior-season variable for y0 mo 1-3
tmppriorObs = climindseasons83Obs['y0 mo 10-12'].loc[firstyear-2:finalyear-1].values
tmppriorDT = climindseasons83DT['y0 mo 10-12'].loc[firstyear-2:finalyear-1].values
# cut to same timeframe as climate data
climindseasons83Obs = climindseasons83Obs.loc[firstyear-1:]
climindseasons83DT = climindseasons83DT.loc[firstyear-1:]
# insert prior-year var
climindseasons83Obs.insert(0, 'y-1 mo 10-12', tmppriorObs)
climindseasons83DT.insert(0, 'y-1 mo 10-12', tmppriorDT)

print('import '+climindnameObs)
print('import '+climindnameDT)

# import average of prior and current-season gSST
climindseasons83Obs_avgdf = pd.read_csv(data_string + 'sstgrad_seasons_' +climindnameObs+'_82_y_avgcurr-prior.txt')
print('import ' +climindnameObs+'_81_y_avgcurr-prior')
climindseasons83DT_avgdf = pd.read_csv(data_string + 'sstgrad_seasons_' +climindnameDT+'_82_y_avgcurr-prior.txt')
print('import ' +climindnameDT+'_81_y_avgcurr-prior')
# filter out columns that don't have y0 or y1 (remove index cols)
climindseasons83Obs_avgdf = climindseasons83Obs_avgdf[[col for col in climindseasons83Obs_avgdf.columns if col.startswith('y0') or col.startswith('y-1')]]
climindseasons83DT_avgdf = climindseasons83DT_avgdf[[col for col in climindseasons83DT_avgdf.columns if col.startswith('y0') or col.startswith('y-1')]]
# remove extra columns
climindseasons83Obs_avgdf = climindseasons83Obs_avgdf.drop(columns=['y-1 mo 1-6','y-1 mo 4-9'])
climindseasons83DT_avgdf = climindseasons83DT_avgdf.drop(columns=['y-1 mo 1-6','y-1 mo 4-9'])
# CHECK: n columns is the same between same-season and averaged climate indices
if len(climindseasons83Obs_avgdf.columns) == len(climindseasons83Obs.columns):
    print('number of seasons present is correct')
else:
    raise RuntimeError("Number of seasons columns is mismatched. Stopping the kernel.")

import patch125-155_nino3-34
import DeTrend_patch125-155_nino3-34
import patch125-155_nino3-34_81_y_avgcurr-prior
import DeTrend_patch125-155_nino3-34_81_y_avgcurr-prior
number of seasons present is correct


# All area

In [7]:
# iterate through each ecoregion, make a list of SST gradients to EXCLUDE

modeloutputfile = model_string + 'coeffExclude_sstavg_all_ecoprovinces.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesall['allwestUS'].columns.values # each climate variable in the columns

for iregion, ecoregname in enumerate(dfnames):
    print('++++++++ ALL model for ' + dfnames[iregion]+ '\t r-value obs \t r-value DT ++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')
        # variables to compare
        # each climate variable
        tmpclim = np.asarray(dfframesall[dfnames[iregion]][label])
        tmpclimDT = annualDetrend(tmpclim)
        # associated concurrent-season climate index (ENSO or other)
        strseason = labels[i].split(' ',1)[1] # get season label
        # match season to the normal 3-month seasons
        matchseason = climindseasons83Obs.columns.isin([strseason]) # True/False array
        imatchseason = np.where(matchseason)[0] # get index
        tmpclimindObs = np.asarray(climindseasons83Obs_avgdf.iloc[:,imatchseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT_avgdf.iloc[:,imatchseason]).flatten()
        
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        # check if they are the same sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[matchseason][0] + 
            '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[imatchseason][0] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[imatchseason][0] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            # use 3-month season name in file output
            f.write(str(climindseasons83Obs.columns[matchseason][0]) + '\n')
f.close()

++++++++ ALL model for allwestUS	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.52322 	SST no trend: r=-0.47438
		 observed SST: p=0.00053188 	SST no trend: p=0.0019845
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.57465 	SST no trend: r=-0.5198
		 observed SST: p=0.00010535 	SST no trend: p=0.00058702
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.29029 	SST no trend: r=-0.18904
		 observed SST: p=0.069201 	SST no trend: p=0.2427
	DT INSIGNIFICANT y0 mo 7-9	p=0.06920 	 p=0.24270
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.2804 	SST no trend: r=-0.23204
		 observed SST: p=0.079684 	SST no trend: p=0.14965
	DT INSIGNIFICANT y0 mo 10-12	p=0.07968 	 p=0.14965
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.41305 	SST no trend: r=0.38185
		 observed SST: p=0.0080733 	SST no trend: p=0.015042
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.38743 	SST no trend: r=0.35551
		 observed SST: p=0.013513 	SST no t

y0 mo 10-12	 observed SST: r=0.28243 	SST no trend: r=0.22819
		 observed SST: p=0.077441 	SST no trend: p=0.15672
	DT INSIGNIFICANT y0 mo 10-12	p=0.07744 	 p=0.15672
PREDICTING prec y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.11461 	SST no trend: r=-0.1164
		 observed SST: p=0.48133 	SST no trend: p=0.47443
	DT INSIGNIFICANT y0 mo 1-3	p=0.48133 	 p=0.47443
PREDICTING prec y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.173 	SST no trend: r=0.15112
		 observed SST: p=0.28575 	SST no trend: p=0.35194
	DT INSIGNIFICANT y0 mo 4-6	p=0.28575 	 p=0.35194
PREDICTING prec y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.23869 	SST no trend: r=-0.28922
		 observed SST: p=0.13799 	SST no trend: p=0.070279
PREDICTING prec y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.2088 	SST no trend: r=0.22878
		 observed SST: p=0.196 	SST no trend: p=0.15562
	DT INSIGNIFICANT y0 mo 10-12	p=0.19600 	 p=0.15562
++++++++ ALL model for ecoprov4	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.30708 	

y0 mo 10-12	 observed SST: r=0.22562 	SST no trend: r=0.15171
		 observed SST: p=0.16157 	SST no trend: p=0.35005
	DT INSIGNIFICANT y0 mo 10-12	p=0.16157 	 p=0.35005
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.30807 	SST no trend: r=0.21996
		 observed SST: p=0.053123 	SST no trend: p=0.17263
	DT INSIGNIFICANT y0 mo 1-3	p=0.05312 	 p=0.17263
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.49855 	SST no trend: r=0.42882
		 observed SST: p=0.0010597 	SST no trend: p=0.0057649
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.33264 	SST no trend: r=0.20773
		 observed SST: p=0.035972 	SST no trend: p=0.19837
	DT INSIGNIFICANT y0 mo 7-9	p=0.03597 	 p=0.19837
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.21402 	SST no trend: r=0.12869
		 observed SST: p=0.18482 	SST no trend: p=0.4287
	DT INSIGNIFICANT y0 mo 10-12	p=0.18482 	 p=0.42870
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.40079 	SST no trend: r=-0.36836
		 observed SST: p=0.01038 	SST no

y0 mo 4-6	 observed SST: r=0.24018 	SST no trend: r=0.22792
		 observed SST: p=0.13549 	SST no trend: p=0.15721
	DT INSIGNIFICANT y0 mo 4-6	p=0.13549 	 p=0.15721
PREDICTING solar y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.050642 	SST no trend: r=0.076546
		 observed SST: p=0.75631 	SST no trend: p=0.63874
	DT INSIGNIFICANT y0 mo 7-9	p=0.75631 	 p=0.63874
PREDICTING solar y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.079184 	SST no trend: r=-0.091102
		 observed SST: p=0.62719 	SST no trend: p=0.57611
	DT INSIGNIFICANT y0 mo 10-12	p=0.62719 	 p=0.57611
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.048785 	SST no trend: r=-0.14189
		 observed SST: p=0.76499 	SST no trend: p=0.38245
	DT INSIGNIFICANT y0 mo 1-3	p=0.76499 	 p=0.38245
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.12324 	SST no trend: r=0.019867
		 observed SST: p=0.44867 	SST no trend: p=0.90315
	DT INSIGNIFICANT y0 mo 4-6	p=0.44867 	 p=0.90315
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.043014 	SST n

y0 mo 1-3	 observed SST: r=0.37324 	SST no trend: r=0.30631
		 observed SST: p=0.017686 	SST no trend: p=0.054562
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.57133 	SST no trend: r=0.51402
		 observed SST: p=0.00011791 	SST no trend: p=0.00069203
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.22176 	SST no trend: r=0.085201
		 observed SST: p=0.16905 	SST no trend: p=0.60117
	DT INSIGNIFICANT y0 mo 7-9	p=0.16905 	 p=0.60117
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.46815 	SST no trend: r=0.42411
		 observed SST: p=0.0023158 	SST no trend: p=0.0063853
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.14139 	SST no trend: r=-0.18584
		 observed SST: p=0.38416 	SST no trend: p=0.25091
	DT INSIGNIFICANT y0 mo 1-3	p=0.38416 	 p=0.25091
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.45402 	SST no trend: r=0.38894
		 observed SST: p=0.0032537 	SST no trend: p=0.013122
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.33683 	SST no trend: 

y0 mo 7-9	 observed SST: r=0.46195 	SST no trend: r=0.39269
		 observed SST: p=0.0026932 	SST no trend: p=0.012196
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.12669 	SST no trend: r=0.098522
		 observed SST: p=0.43599 	SST no trend: p=0.54529
	DT INSIGNIFICANT y0 mo 10-12	p=0.43599 	 p=0.54529
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.11716 	SST no trend: r=-0.083818
		 observed SST: p=0.47155 	SST no trend: p=0.60711
	DT INSIGNIFICANT y0 mo 1-3	p=0.47155 	 p=0.60711
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.41023 	SST no trend: r=-0.39833
		 observed SST: p=0.0085607 	SST no trend: p=0.010905
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.35679 	SST no trend: r=-0.29603
		 observed SST: p=0.023823 	SST no trend: p=0.063638
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.14625 	SST no trend: r=0.17069
		 observed SST: p=0.36784 	SST no trend: p=0.29232
	DT INSIGNIFICANT y0 mo 10-12	p=0.36784 	 p=0.29232
PREDICTIN

y0 mo 4-6	 observed SST: r=0.29393 	SST no trend: r=0.23974
		 observed SST: p=0.06563 	SST no trend: p=0.13623
	DT INSIGNIFICANT y0 mo 4-6	p=0.06563 	 p=0.13623
PREDICTING tmean y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.38455 	SST no trend: r=0.2906
		 observed SST: p=0.014286 	SST no trend: p=0.068894
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.29961 	SST no trend: r=0.24618
		 observed SST: p=0.060349 	SST no trend: p=0.1257
	DT INSIGNIFICANT y0 mo 10-12	p=0.06035 	 p=0.12570
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.30244 	SST no trend: r=0.22123
		 observed SST: p=0.057849 	SST no trend: p=0.17009
	DT INSIGNIFICANT y0 mo 1-3	p=0.05785 	 p=0.17009
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.47961 	SST no trend: r=0.41396
		 observed SST: p=0.0017395 	SST no trend: p=0.007921
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.37784 	SST no trend: r=0.27898
		 observed SST: p=0.016227 	SST no trend: p=0.081282
PREDICTING vpd y0 mo 10-12
y0 mo 10-

y0 mo 4-6	 observed SST: r=0.095295 	SST no trend: r=0.11696
		 observed SST: p=0.5586 	SST no trend: p=0.47231
	DT INSIGNIFICANT y0 mo 4-6	p=0.55860 	 p=0.47231
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.18003 	SST no trend: r=-0.13933
		 observed SST: p=0.26632 	SST no trend: p=0.39119
	DT INSIGNIFICANT y0 mo 7-9	p=0.26632 	 p=0.39119
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.21348 	SST no trend: r=0.19711
		 observed SST: p=0.18594 	SST no trend: p=0.2228
	DT INSIGNIFICANT y0 mo 10-12	p=0.18594 	 p=0.22280
PREDICTING wind y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.055933 	SST no trend: r=-0.053053
		 observed SST: p=0.73175 	SST no trend: p=0.74508
	DT INSIGNIFICANT y0 mo 1-3	p=0.73175 	 p=0.74508
PREDICTING wind y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.23771 	SST no trend: r=0.22161
		 observed SST: p=0.13968 	SST no trend: p=0.16936
	DT INSIGNIFICANT y0 mo 4-6	p=0.13968 	 p=0.16936
PREDICTING wind y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.030957 	SST no

y0 mo 1-3	 observed SST: r=0.56738 	SST no trend: r=0.52165
		 observed SST: p=0.00013461 	SST no trend: p=0.00055657
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.43832 	SST no trend: r=0.35514
		 observed SST: p=0.00467 	SST no trend: p=0.024527
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.085928 	SST no trend: r=-0.016016
		 observed SST: p=0.59806 	SST no trend: p=0.92186
	SIGN CHANGE y0 mo 7-9	r=0.08593 	 r=-0.01602
	DT INSIGNIFICANT y0 mo 7-9	p=0.59806 	 p=0.92186
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.50493 	SST no trend: r=0.46842
		 observed SST: p=0.00089121 	SST no trend: p=0.0023005
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.38557 	SST no trend: r=-0.34128
		 observed SST: p=0.014008 	SST no trend: p=0.031146
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.20955 	SST no trend: r=-0.10065
		 observed SST: p=0.19436 	SST no trend: p=0.53659
	DT INSIGNIFICANT y0 mo 4-6	p=0.19436 	 p=0.53659
PREDICTING wetdays y0 mo

# Forest

In [8]:
# iterate through each ecoregion, make a list of SST gradients to EXCLUDE

modeloutputfile = model_string + 'coeffExclude_sstavg_for_ecoprovinces.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesfor['allwestUS'].columns.values # each climate variable in the columns

for iregion, ecoregname in enumerate(dfnames):
    print('++++++++ FOREST model for ' + dfnames[iregion]+ '\t r-value obs \t r-value DT ++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')
        # variables to compare
        # each climate variable
        tmpclim = np.asarray(dfframesfor[dfnames[iregion]][label])
        tmpclimDT = annualDetrend(tmpclim)
        # associated concurrent-season climate index (ENSO or other)
        strseason = labels[i].split(' ',1)[1] # get season label
        # match to 3-month seasons names
        matchseason = climindseasons83Obs.columns.isin([strseason]) # True/False array
        imatchseason = np.where(matchseason)[0] # get index
        tmpclimindObs = np.asarray(climindseasons83Obs_avgdf.iloc[:,imatchseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT_avgdf.iloc[:,imatchseason]).flatten()
        
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        # check if they are the same sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[matchseason][0] + 
            '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[imatchseason][0] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[imatchseason][0] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            # use 3-month season in file name
            f.write(str(climindseasons83Obs.columns[matchseason][0]) + '\n')
f.close()

++++++++ FOREST model for allwestUS	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.49343 	SST no trend: r=-0.43505
		 observed SST: p=0.0012149 	SST no trend: p=0.0050246
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.51146 	SST no trend: r=-0.43877
		 observed SST: p=0.00074355 	SST no trend: p=0.0046225
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.28496 	SST no trend: r=-0.17402
		 observed SST: p=0.074711 	SST no trend: p=0.28285
	DT INSIGNIFICANT y0 mo 7-9	p=0.07471 	 p=0.28285
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.20741 	SST no trend: r=-0.14672
		 observed SST: p=0.19906 	SST no trend: p=0.36632
	DT INSIGNIFICANT y0 mo 10-12	p=0.19906 	 p=0.36632
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.2669 	SST no trend: r=0.24704
		 observed SST: p=0.095941 	SST no trend: p=0.12434
	DT INSIGNIFICANT y0 mo 1-3	p=0.09594 	 p=0.12434
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.28309 	SST no tren

y0 mo 1-3	 observed SST: r=-0.36463 	SST no trend: r=-0.34545
		 observed SST: p=0.020708 	SST no trend: p=0.029018
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.069839 	SST no trend: r=-0.028695
		 observed SST: p=0.66849 	SST no trend: p=0.86048
	DT INSIGNIFICANT y0 mo 4-6	p=0.66849 	 p=0.86048
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.22914 	SST no trend: r=0.13437
		 observed SST: p=0.15496 	SST no trend: p=0.40843
	DT INSIGNIFICANT y0 mo 7-9	p=0.15496 	 p=0.40843
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.055814 	SST no trend: r=-0.12776
		 observed SST: p=0.7323 	SST no trend: p=0.4321
	DT INSIGNIFICANT y0 mo 10-12	p=0.73230 	 p=0.43210
PREDICTING tmean y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.31752 	SST no trend: r=-0.30402
		 observed SST: p=0.045882 	SST no trend: p=0.056492
PREDICTING tmean y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.011223 	SST no trend: r=0.043469
		 observed SST: p=0.9452 	SST no trend: p=0.78998
	SIGN CHANGE y0 mo 4-6	r=

y0 mo 7-9	 observed SST: r=0.053998 	SST no trend: r=-0.17629
		 observed SST: p=0.7407 	SST no trend: p=0.27652
	SIGN CHANGE y0 mo 7-9	r=0.05400 	 r=-0.17629
	DT INSIGNIFICANT y0 mo 7-9	p=0.74070 	 p=0.27652
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.073407 	SST no trend: r=-0.14325
		 observed SST: p=0.6526 	SST no trend: p=0.37786
	DT INSIGNIFICANT y0 mo 10-12	p=0.65260 	 p=0.37786
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.13255 	SST no trend: r=0.11579
		 observed SST: p=0.41488 	SST no trend: p=0.47679
	DT INSIGNIFICANT y0 mo 1-3	p=0.41488 	 p=0.47679
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.16961 	SST no trend: r=0.10422
		 observed SST: p=0.29543 	SST no trend: p=0.52216
	DT INSIGNIFICANT y0 mo 4-6	p=0.29543 	 p=0.52216
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.14195 	SST no trend: r=0.034863
		 observed SST: p=0.38225 	SST no trend: p=0.83089
	DT INSIGNIFICANT y0 mo 7-9	p=0.38225 	 p=0.83089
PREDICTING vpd y0 mo 10-12
y0 mo

y0 mo 7-9	 observed SST: r=0.2796 	SST no trend: r=0.14774
		 observed SST: p=0.080581 	SST no trend: p=0.36294
	DT INSIGNIFICANT y0 mo 7-9	p=0.08058 	 p=0.36294
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.06199 	SST no trend: r=-0.0073862
		 observed SST: p=0.70395 	SST no trend: p=0.96392
	SIGN CHANGE y0 mo 10-12	r=0.06199 	 r=-0.00739
	DT INSIGNIFICANT y0 mo 10-12	p=0.70395 	 p=0.96392
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.36256 	SST no trend: r=-0.33928
		 observed SST: p=0.021494 	SST no trend: p=0.032215
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.29033 	SST no trend: r=-0.2714
		 observed SST: p=0.069168 	SST no trend: p=0.090262
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.066919 	SST no trend: r=0.1655
		 observed SST: p=0.68161 	SST no trend: p=0.30747
	DT INSIGNIFICANT y0 mo 7-9	p=0.68161 	 p=0.30747
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.073904 	SST no trend: r=-0.062733
		 observed SST: 

y0 mo 7-9	 observed SST: r=0.38204 	SST no trend: r=0.33179
		 observed SST: p=0.014987 	SST no trend: p=0.036478
PREDICTING solar y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.00066439 	SST no trend: r=-0.0054144
		 observed SST: p=0.99675 	SST no trend: p=0.97355
	SIGN CHANGE y0 mo 10-12	r=0.00066 	 r=-0.00541
	DT INSIGNIFICANT y0 mo 10-12	p=0.99675 	 p=0.97355
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.23186 	SST no trend: r=-0.2655
		 observed SST: p=0.14998 	SST no trend: p=0.097758
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.11829 	SST no trend: r=0.097385
		 observed SST: p=0.46725 	SST no trend: p=0.54997
	DT INSIGNIFICANT y0 mo 4-6	p=0.46725 	 p=0.54997
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.32564 	SST no trend: r=0.22391
		 observed SST: p=0.040315 	SST no trend: p=0.16486
	DT INSIGNIFICANT y0 mo 7-9	p=0.04032 	 p=0.16486
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.13832 	SST no trend: r=0.064554
		 observed SST: p=0.39469 	

y0 mo 10-12	 observed SST: r=0.10106 	SST no trend: r=0.1393
		 observed SST: p=0.53493 	SST no trend: p=0.3913
	DT INSIGNIFICANT y0 mo 10-12	p=0.53493 	 p=0.39130
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.15668 	SST no trend: r=-0.1224
		 observed SST: p=0.3343 	SST no trend: p=0.45182
	DT INSIGNIFICANT y0 mo 1-3	p=0.33430 	 p=0.45182
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.12872 	SST no trend: r=0.14752
		 observed SST: p=0.42862 	SST no trend: p=0.36367
	DT INSIGNIFICANT y0 mo 4-6	p=0.42862 	 p=0.36367
PREDICTING solar y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.442 	SST no trend: r=0.40355
		 observed SST: p=0.0042966 	SST no trend: p=0.0098169
PREDICTING solar y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.32616 	SST no trend: r=-0.30156
		 observed SST: p=0.039981 	SST no trend: p=0.058619
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.4392 	SST no trend: r=-0.49394
		 observed SST: p=0.0045778 	SST no trend: p=0.0011986
PREDICTING tmax y0 mo 4-6
y

y0 mo 10-12	 observed SST: r=0.34687 	SST no trend: r=0.33601
		 observed SST: p=0.028321 	SST no trend: p=0.034024
++++++++ FOREST model for ecoprov16	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.45018 	SST no trend: r=-0.39046
		 observed SST: p=0.0035598 	SST no trend: p=0.012741
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.55552 	SST no trend: r=-0.50489
		 observed SST: p=0.00019844 	SST no trend: p=0.00089206
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.13482 	SST no trend: r=-0.058192
		 observed SST: p=0.40686 	SST no trend: p=0.72134
	DT INSIGNIFICANT y0 mo 7-9	p=0.40686 	 p=0.72134
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.28958 	SST no trend: r=-0.24073
		 observed SST: p=0.06992 	SST no trend: p=0.13456
	DT INSIGNIFICANT y0 mo 10-12	p=0.06992 	 p=0.13456
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.4477 	SST no trend: r=0.40939
		 observed SST: p=0.003771 	SST no trend: p=0.0087119
PREDIC

y0 mo 10-12	 observed SST: r=0.099023 	SST no trend: r=0.042373
		 observed SST: p=0.54325 	SST no trend: p=0.79517
	DT INSIGNIFICANT y0 mo 10-12	p=0.54325 	 p=0.79517
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.17578 	SST no trend: r=0.11408
		 observed SST: p=0.27794 	SST no trend: p=0.48334
	DT INSIGNIFICANT y0 mo 1-3	p=0.27794 	 p=0.48334
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.30377 	SST no trend: r=0.21592
		 observed SST: p=0.056705 	SST no trend: p=0.18084
	DT INSIGNIFICANT y0 mo 4-6	p=0.05670 	 p=0.18084
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.28168 	SST no trend: r=0.14589
		 observed SST: p=0.078262 	SST no trend: p=0.36906
	DT INSIGNIFICANT y0 mo 7-9	p=0.07826 	 p=0.36906
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.11301 	SST no trend: r=0.055427
		 observed SST: p=0.48748 	SST no trend: p=0.73408
	DT INSIGNIFICANT y0 mo 10-12	p=0.48748 	 p=0.73408
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.27761 	SST no tr

# Nonforest

In [9]:
# iterate through each ecoregion, make a list of SST gradients to EXCLUDE

modeloutputfile = model_string + 'coeffExclude_sstavg_non_ecoprovinces.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesnon['allwestUS'].columns.values # each climate variable in the columns

for iregion, ecoregname in enumerate(dfnames):
    print('++++++++ NONFOREST model for ' + dfnames[iregion]+ '\t r-value obs \t r-value DT ++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')
        # variables to compare
        # each climate variable
        tmpclim = np.asarray(dfframesnon[dfnames[iregion]][label])
        tmpclimDT = annualDetrend(tmpclim)
        # associated concurrent-season climate index (ENSO or other)
        strseason = labels[i].split(' ',1)[1] # get season label
        # match name to 3-month season name
        matchseason = climindseasons83Obs.columns.isin([strseason]) # True/False array
        imatchseason = np.where(matchseason)[0] # get index
        tmpclimindObs = np.asarray(climindseasons83Obs_avgdf.iloc[:,imatchseason]).flatten()
        tmpclimindDT = np.asarray(climindseasons83DT_avgdf.iloc[:,imatchseason]).flatten()
        
        # correlation of clim vs. SST gradient
        tmppearsonObs = pearsonr(tmpclimindObs, tmpclim)
        tmprObs = tmppearsonObs.statistic
        tmppObs = tmppearsonObs.pvalue
        tmppearsonDT = pearsonr(tmpclimindDT, tmpclimDT)
        tmprDT = tmppearsonDT.statistic
        tmppDT = tmppearsonDT.pvalue
        # check if they are the same sign
        tmpsign = ((tmprObs > 0) == (tmprDT > 0))
        tmppvalue = (tmppDT < 0.1) # check significance
        print(climindseasons83Obs.columns[matchseason][0] + 
            '\t observed SST: r={:.5} \tSST no trend: r={:.5}'.format(tmprObs, tmprDT))
        print('\t\t observed SST: p={:.5} \tSST no trend: p={:.5}'.format(tmppObs, tmppDT))
        
        if tmpsign==False:
            print('\tSIGN CHANGE ' + climindseasons83Obs.columns[imatchseason][0] + '\t'+
                  'r={:.5f} \t r={:.5f}'.format(tmprObs, tmprDT))
        if tmppvalue==False:
            print('\tDT INSIGNIFICANT ' + climindseasons83Obs.columns[imatchseason][0] + '\t'+
                  'p={:.5f} \t p={:.5f}'.format(tmppObs, tmppDT))
        if (tmpsign==False) or (tmppvalue==False):
            # use 3-month season name in file
            f.write(str(climindseasons83Obs.columns[matchseason][0]) + '\n')
f.close()

++++++++ NONFOREST model for allwestUS	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.52169 	SST no trend: r=-0.47579
		 observed SST: p=0.00055586 	SST no trend: p=0.0019155
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.58937 	SST no trend: r=-0.54112
		 observed SST: p=6.2975e-05 	SST no trend: p=0.00031182
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.28885 	SST no trend: r=-0.193
		 observed SST: p=0.070663 	SST no trend: p=0.23279
	DT INSIGNIFICANT y0 mo 7-9	p=0.07066 	 p=0.23279
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.30114 	SST no trend: r=-0.25755
		 observed SST: p=0.058988 	SST no trend: p=0.10862
	DT INSIGNIFICANT y0 mo 10-12	p=0.05899 	 p=0.10862
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=0.46053 	SST no trend: r=0.42676
		 observed SST: p=0.0027869 	SST no trend: p=0.0060292
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.4115 	SST no trend: r=0.38549
		 observed SST: p=0.0083392 	

y0 mo 7-9	 observed SST: r=0.26612 	SST no trend: r=0.25791
		 observed SST: p=0.096957 	SST no trend: p=0.10811
	DT INSIGNIFICANT y0 mo 7-9	p=0.09696 	 p=0.10811
PREDICTING solar y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.1035 	SST no trend: r=0.089375
		 observed SST: p=0.52509 	SST no trend: p=0.58339
	DT INSIGNIFICANT y0 mo 10-12	p=0.52509 	 p=0.58339
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.022593 	SST no trend: r=-0.10612
		 observed SST: p=0.88994 	SST no trend: p=0.51458
	DT INSIGNIFICANT y0 mo 1-3	p=0.88994 	 p=0.51458
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.31292 	SST no trend: r=0.26275
		 observed SST: p=0.049298 	SST no trend: p=0.10143
	DT INSIGNIFICANT y0 mo 4-6	p=0.04930 	 p=0.10143
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.3456 	SST no trend: r=0.23282
		 observed SST: p=0.028946 	SST no trend: p=0.14824
	DT INSIGNIFICANT y0 mo 7-9	p=0.02895 	 p=0.14824
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.27327 	SST no t

y0 mo 1-3	 observed SST: r=-0.049991 	SST no trend: r=-0.083507
		 observed SST: p=0.75935 	SST no trend: p=0.60844
	DT INSIGNIFICANT y0 mo 1-3	p=0.75935 	 p=0.60844
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.085813 	SST no trend: r=0.12408
		 observed SST: p=0.59854 	SST no trend: p=0.44557
	DT INSIGNIFICANT y0 mo 4-6	p=0.59854 	 p=0.44557
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.35636 	SST no trend: r=0.32884
		 observed SST: p=0.024008 	SST no trend: p=0.038279
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.14737 	SST no trend: r=0.11184
		 observed SST: p=0.36416 	SST no trend: p=0.49203
	DT INSIGNIFICANT y0 mo 10-12	p=0.36416 	 p=0.49203
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.096038 	SST no trend: r=-0.097769
		 observed SST: p=0.55552 	SST no trend: p=0.54839
	DT INSIGNIFICANT y0 mo 1-3	p=0.55552 	 p=0.54839
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.086289 	SST no trend: r=0.056772
		 observed SST: p=0.59651 	

y0 mo 1-3	 observed SST: r=-0.37916 	SST no trend: r=-0.3394
		 observed SST: p=0.01583 	SST no trend: p=0.032151
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.27957 	SST no trend: r=-0.28745
		 observed SST: p=0.080617 	SST no trend: p=0.072099
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.097353 	SST no trend: r=0.19986
		 observed SST: p=0.5501 	SST no trend: p=0.21628
	DT INSIGNIFICANT y0 mo 7-9	p=0.55010 	 p=0.21628
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.099439 	SST no trend: r=-0.091218
		 observed SST: p=0.54154 	SST no trend: p=0.57563
	DT INSIGNIFICANT y0 mo 10-12	p=0.54154 	 p=0.57563
PREDICTING wind y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.087877 	SST no trend: r=-0.10127
		 observed SST: p=0.58975 	SST no trend: p=0.53407
	DT INSIGNIFICANT y0 mo 1-3	p=0.58975 	 p=0.53407
PREDICTING wind y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.41251 	SST no trend: r=0.38603
		 observed SST: p=0.0081652 	SST no trend: p=0.013883
PREDICTING wind y

y0 mo 7-9	 observed SST: r=-0.33513 	SST no trend: r=-0.27732
		 observed SST: p=0.034523 	SST no trend: p=0.083191
PREDICTING prec y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.18227 	SST no trend: r=0.18638
		 observed SST: p=0.26031 	SST no trend: p=0.24952
	DT INSIGNIFICANT y0 mo 10-12	p=0.26031 	 p=0.24952
++++++++ NONFOREST model for ecoprov14	 r-value obs 	 r-value DT ++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.44494 	SST no trend: r=-0.3901
		 observed SST: p=0.0040178 	SST no trend: p=0.01283
PREDICTING rh y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.52835 	SST no trend: r=-0.47362
		 observed SST: p=0.00045776 	SST no trend: p=0.0020225
PREDICTING rh y0 mo 7-9
y0 mo 7-9	 observed SST: r=-0.21392 	SST no trend: r=-0.11444
		 observed SST: p=0.18502 	SST no trend: p=0.48196
	DT INSIGNIFICANT y0 mo 7-9	p=0.18502 	 p=0.48196
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.24939 	SST no trend: r=-0.20904
		 observed SST: p=0.12068 	SST no trend: p=0.19548
	DT IN

y0 mo 7-9	 observed SST: r=-0.075985 	SST no trend: r=-0.044893
		 observed SST: p=0.64121 	SST no trend: p=0.78327
	DT INSIGNIFICANT y0 mo 7-9	p=0.64121 	 p=0.78327
PREDICTING rh y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.25543 	SST no trend: r=0.21052
		 observed SST: p=0.11167 	SST no trend: p=0.19228
	DT INSIGNIFICANT y0 mo 10-12	p=0.11167 	 p=0.19228
PREDICTING solar y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.12167 	SST no trend: r=-0.11148
		 observed SST: p=0.45452 	SST no trend: p=0.49344
	DT INSIGNIFICANT y0 mo 1-3	p=0.45452 	 p=0.49344
PREDICTING solar y0 mo 4-6
y0 mo 4-6	 observed SST: r=-0.24465 	SST no trend: r=-0.26847
		 observed SST: p=0.12814 	SST no trend: p=0.093928
PREDICTING solar y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.1047 	SST no trend: r=0.065804
		 observed SST: p=0.52024 	SST no trend: p=0.68664
	DT INSIGNIFICANT y0 mo 7-9	p=0.52024 	 p=0.68664
PREDICTING solar y0 mo 10-12
y0 mo 10-12	 observed SST: r=-0.23944 	SST no trend: r=-0.23338
		 observed SST: p=0.13673 

y0 mo 4-6	 observed SST: r=0.27455 	SST no trend: r=0.2239
		 observed SST: p=0.086443 	SST no trend: p=0.16488
	DT INSIGNIFICANT y0 mo 4-6	p=0.08644 	 p=0.16488
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.40469 	SST no trend: r=0.31635
		 observed SST: p=0.0095934 	SST no trend: p=0.046729
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	 observed SST: r=0.34439 	SST no trend: r=0.2795
		 observed SST: p=0.029548 	SST no trend: p=0.080693
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	 observed SST: r=-0.22374 	SST no trend: r=-0.27874
		 observed SST: p=0.16519 	SST no trend: p=0.081558
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	 observed SST: r=0.10155 	SST no trend: r=0.061261
		 observed SST: p=0.53297 	SST no trend: p=0.70728
	DT INSIGNIFICANT y0 mo 4-6	p=0.53297 	 p=0.70728
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	 observed SST: r=0.288 	SST no trend: r=0.17149
		 observed SST: p=0.071527 	SST no trend: p=0.29003
	DT INSIGNIFICANT y0 mo 7-9	p=0.07153 	 p=0.29003
PREDICTING tmin y0 mo 10-12
y0 mo 10-

In [10]:
# print last time this checker was updated
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Last checked correlations =", dt_string)
print("SST gradient =", climindnameObs)

Last checked correlations = 26/06/2026 14:54:24
SST gradient = patch125-155_nino3-34
